In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
results = pd.read_csv(CLEANED_DATA / "results_2000_onwards.csv")


In [2]:
team_results = team_matches.groupby("team")["result"].value_counts().unstack(fill_value = 0)

team_results.head()

result,D,L,W
team,,,
Abkhazia,13,5,15
Afghanistan,31,58,35
Albania,55,107,86
Alderney,2,42,3
Algeria,74,75,156


In [3]:
matches_played = team_matches.groupby("team").size()
matches_played.head()

team
Abkhazia        33
Afghanistan    124
Albania        248
Alderney        47
Algeria        305
dtype: int64

In [4]:
goals_for = team_matches.groupby("team")["goals_for"].sum()
goals_for.head()

team
Abkhazia        54
Afghanistan    129
Albania        263
Alderney        31
Algeria        518
Name: goals_for, dtype: int64

In [5]:
goals_against = team_matches.groupby("team")["goals_against"].sum()
goals_against.head()

team
Abkhazia        26
Afghanistan    219
Albania        291
Alderney       191
Algeria        305
Name: goals_against, dtype: int64

In [6]:
team_stats = team_results.copy()

team_stats["matches_played"] = matches_played
team_stats["goals_for"] = goals_for
team_stats["goals_against"] = goals_against
team_stats.head()

result,D,L,W,matches_played,goals_for,goals_against
team,,,,,,
Abkhazia,13,5,15,33,54,26
Afghanistan,31,58,35,124,129,219
Albania,55,107,86,248,263,291
Alderney,2,42,3,47,31,191
Algeria,74,75,156,305,518,305


In [9]:
team_stats["goal_difference"] = (team_stats["goals_for"] - team_stats["goals_against"])

team_stats["win_pct"] = (team_stats["W"] / team_stats["matches_played"] * 100).round(2)

team_stats.head()

result,D,L,W,matches_played,goals_for,goals_against,goal_difference,win_pct
team,,,,,,,,
Abkhazia,13,5,15,33,54,26,28,45.45
Afghanistan,31,58,35,124,129,219,-90,28.23
Albania,55,107,86,248,263,291,-28,34.68
Alderney,2,42,3,47,31,191,-160,6.38
Algeria,74,75,156,305,518,305,213,51.15


In [10]:
(team_stats["matches_played"] == team_stats["W"] + team_stats["D"] + team_stats["L"]).all()
team_stats.loc["Brazil"]

result
D                   80.0
L                   55.0
W                  240.0
matches_played     375.0
goals_for          783.0
goals_against      269.0
goal_difference    514.0
win_pct             64.0
Name: Brazil, dtype: float64

In [11]:
team_stats = team_stats[["matches_played", "W", "D", "L", "goals_for", "goals_against", "goal_difference", "win_pct"]]
team_stats.head()

result,matches_played,W,D,L,goals_for,goals_against,goal_difference,win_pct
team,,,,,,,,
Abkhazia,33,15,13,5,54,26,28,45.45
Afghanistan,124,35,31,58,129,219,-90,28.23
Albania,248,86,55,107,263,291,-28,34.68
Alderney,47,3,2,42,31,191,-160,6.38
Algeria,305,156,74,75,518,305,213,51.15


In [12]:
team_stats.to_csv(PROCESSED_DATA / "team_stats.csv")

Completed: Team Statistics Engineering

Generated team_stats.csv from team_matches.csv containing aggregated team-level performance metrics since 2000, including wins, draws, losses, goals scored, goals conceded, goal difference, and win percentage. Validation checks confirmed internal consistency (W + D + L = matches_played) and spot checks on major teams produced realistic results